In [0]:
import geopandas as gpd
from shapely.ops import unary_union

# --------------------------------------------------
# 1. Set your file paths
# --------------------------------------------------

large_path = "/dbfs/FileStore/AndyBeverton/33035/33035.shp"
subs_path  = "/dbfs/FileStore/AndyBeverton/UnionCatchments/DenverAOI_union_contributingcatchments.shp"

output_path = "/dbfs/FileStore/AndyBeverton/33035_minus_contributingcatchments.shp"



# --------------------------------------------------
# 2. Read shapefiles
# --------------------------------------------------

large = gpd.read_file(large_path)
subs = gpd.read_file(subs_path)

# --------------------------------------------------
# 2b. Check inputs before processing
# --------------------------------------------------

print("Large rows:", len(large))
print("Subs rows:", len(subs))

print("Large CRS:", large.crs)
print("Subs CRS:", subs.crs)

print("Large geometry types:")
print(large.geometry.geom_type.value_counts())

print("Subs geometry types:")
print(subs.geometry.geom_type.value_counts())

print("Large bounds:")
print(large.total_bounds)

print("Subs bounds:")
print(subs.total_bounds)

print("Empty geometries in subs:", subs.geometry.is_empty.sum())
print("Null geometries in subs:", subs.geometry.isna().sum())

# --------------------------------------------------
# 3. Make sure they use the same CRS
# --------------------------------------------------

subs = subs.to_crs(large.crs)

# Use British National Grid for metre-based buffering
large = large.to_crs("EPSG:27700")
subs = subs.to_crs("EPSG:27700")

# --------------------------------------------------
# 3b. Check after CRS conversion
# --------------------------------------------------

print("After CRS conversion")
print("Large CRS:", large.crs)
print("Subs CRS:", subs.crs)

print("Large bounds:")
print(large.total_bounds)

print("Subs bounds:")
print(subs.total_bounds)

print("Do they overlap?")
print(large.unary_union.intersects(subs.unary_union))

# --------------------------------------------------
# 4. Fix invalid geometries
# --------------------------------------------------

large["geometry"] = large.geometry.make_valid()
subs["geometry"] = subs.geometry.make_valid()

# --------------------------------------------------
# 5. Dissolve subcatchments and buffer slightly
# --------------------------------------------------

buffer_distance_m = 10  # try 5, 10, 25 depending on slivers

subs_union = unary_union(subs.geometry)
subs_union_buffered = subs_union.buffer(buffer_distance_m)

# --------------------------------------------------
# 6. Remove subcatchments from the large catchment
# --------------------------------------------------

large_union = unary_union(large.geometry)

remaining_geom = large_union.difference(subs_union_buffered)
# --------------------------------------------------
# 7. Split into individual polygons and keep largest
# --------------------------------------------------

remaining = gpd.GeoDataFrame(geometry=[remaining_geom], crs="EPSG:27700")

# Split multipart geometry into separate polygon rows
remaining = remaining.explode(index_parts=False).reset_index(drop=True)

# Remove empty/null geometries
remaining = remaining[
    (~remaining.geometry.is_empty) &
    (remaining.geometry.notna())
].copy()

# Calculate area of each polygon
remaining["area_m2"] = remaining.area

# Print areas so you can check what is being removed
print("Number of polygons after difference:", len(remaining))
print(
    remaining[["area_m2"]]
    .sort_values("area_m2", ascending=False)
    .head(20)
)

# Optional: save all remaining polygons before filtering
all_parts_output_path = "/dbfs/FileStore/AndyBeverton/33035_minus_contributingcatchments_ALL_PARTS.shp"
remaining.to_file(all_parts_output_path, driver="ESRI Shapefile")

print("Saved all exploded parts to:")
print(all_parts_output_path)

# Keep only the single largest polygon
remaining_main = (
    remaining
    .sort_values("area_m2", ascending=False)
    .head(1)
    .drop(columns="area_m2")
    .copy()
)
# --------------------------------------------------
# 8. Save output
# --------------------------------------------------

# --------------------------------------------------
# 8. Save output to clean folder
# --------------------------------------------------

import os
import shutil

output_folder = "/dbfs/FileStore/AndyBeverton/33035_minus_contributingcatchments_out"
output_path = f"{output_folder}/33035_minus_contributingcatchments.shp"

# Delete old output folder if it exists
if os.path.exists(output_folder):
    shutil.rmtree(output_folder)

os.makedirs(output_folder, exist_ok=True)

remaining_main.to_file(output_path, driver="ESRI Shapefile")

print("Saved cleaned catchment to:")
print(output_path)

# --------------------------------------------------
# 8b. Check saved shapefile was written correctly
# --------------------------------------------------

import os

output_base = "/dbfs/FileStore/AndyBeverton/33035_minus_contributingcatchments_out/33035_minus_contributingcatchments"

expected_files = [".shp", ".shx", ".dbf", ".prj", ".cpg"]

print("\nChecking shapefile components:")
for ext in expected_files:
    f = output_base + ext
    if os.path.exists(f):
        print(f"✅ {ext} exists | size: {os.path.getsize(f):,} bytes")
    else:
        print(f"❌ {ext} missing")

# Read the saved shapefile back in
saved = gpd.read_file(output_base + ".shp")

print("\nSaved shapefile check:")
print("Rows:", len(saved))
print("CRS:", saved.crs)
print("Geometry types:")
print(saved.geometry.geom_type.value_counts())
print("Bounds:")
print(saved.total_bounds)
print("Area m2:")
print(saved.area.sum())

print("\nFirst rows:")
print(saved.head())

# Compare with original remaining_main before saving
print("\nOriginal output before save:")
print("Rows:", len(remaining_main))
print("CRS:", remaining_main.crs)
print("Bounds:", remaining_main.total_bounds)
print("Area m2:", remaining_main.area.sum())

print("\nSaved output after reload:")
print("Rows:", len(saved))
print("CRS:", saved.crs)
print("Bounds:", saved.total_bounds)
print("Area m2:", saved.area.sum())


# --------------------------------------------------
# 9. Plot inputs and output for checking
# --------------------------------------------------

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

fig = plt.figure(figsize=(18, 14))

# Top row: individual plots
ax1 = plt.subplot2grid((2, 3), (0, 0))
ax2 = plt.subplot2grid((2, 3), (0, 1))
ax3 = plt.subplot2grid((2, 3), (0, 2))

# Bottom row: combined plot across full width
ax4 = plt.subplot2grid((2, 3), (1, 0), colspan=3)

# Individual plot 1: large catchment
large.plot(
    ax=ax1,
    facecolor="none",
    edgecolor="black",
    linewidth=1.5
)
ax1.set_title("Input 1: 33035 large catchment")
ax1.set_aspect("equal")

# Individual plot 2: subcatchments
subs.plot(
    ax=ax2,
    facecolor="none",
    edgecolor="red",
    linewidth=1.2,
    linestyle="--"
)
ax2.set_title("Input 2: AOIcliptest subcatchments")
ax2.set_aspect("equal")

# Individual plot 3: output
remaining_main.plot(
    ax=ax3,
    facecolor="none",
    edgecolor="green",
    linewidth=1.5
)
ax3.set_title("Output: 33035 minus AOIcliptest")
ax3.set_aspect("equal")

# Combined plot
large.plot(
    ax=ax4,
    facecolor="lightgrey",
    edgecolor="black",
    linewidth=1.5,
    alpha=0.25
)

subs.plot(
    ax=ax4,
    facecolor="red",
    edgecolor="red",
    linewidth=1.0,
    alpha=0.35,
    linestyle="--"
)

remaining_main.plot(
    ax=ax4,
    facecolor="none",
    edgecolor="green",
    linewidth=2.0,
    linestyle="-"
)

ax4.set_title("Combined check: original catchment, removed subcatchments, and final output")
ax4.set_aspect("equal")

# Custom legend
legend_items = [
    Line2D([0], [0], color="black", linewidth=1.5, label="33035 original catchment"),
    Line2D([0], [0], color="red", linewidth=1.5, linestyle="--", label="AOIcliptest removed subcatchments"),
    Line2D([0], [0], color="green", linewidth=2.0, label="Final output"),
]

ax4.legend(handles=legend_items, loc="best")

# Remove axis clutter
for ax in [ax1, ax2, ax3, ax4]:
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.ticklabel_format(useOffset=False)
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()


In [0]:
dbutils.fs.rm(
    "dbfs:/FileStore/AndyBeverton/UnionCatchments",
    recurse=True
)

In [0]:
# ==================================================
# Remove contributing catchments from 33035 catchment
# ==================================================

import os
import shutil
import geopandas as gpd
import matplotlib.pyplot as plt

from shapely.ops import unary_union
from matplotlib.lines import Line2D


# --------------------------------------------------
# 1. User settings
# --------------------------------------------------

BASE_DIR = "/dbfs/FileStore/AndyBeverton"

LARGE_PATH = f"{BASE_DIR}/33035/33035.shp"
SUBS_PATH = f"{BASE_DIR}/Contributingcatchments/Contributing_Catchments.shp"

OUTPUT_FOLDER = f"{BASE_DIR}/AOI"
OUTPUT_NAME = "33035_minus_contributingcatchments"
OUTPUT_PATH = f"{OUTPUT_FOLDER}/{OUTPUT_NAME}.shp"

ALL_PARTS_FOLDER = f"{BASE_DIR}/ALL PARTS"
ALL_PARTS_PATH = f"{ALL_PARTS_FOLDER}/{OUTPUT_NAME}_ALL_PARTS.shp"

TARGET_CRS = "EPSG:27700"
BUFFER_DISTANCE_M = 10  # try 0, 5, 10, 25, 50


# --------------------------------------------------
# 2. Helper functions
# --------------------------------------------------

def check_gdf(gdf, name):
    print(f"\n--- {name} ---")
    print("Rows:", len(gdf))
    print("CRS:", gdf.crs)
    print("Geometry types:")
    print(gdf.geometry.geom_type.value_counts())
    print("Bounds:", gdf.total_bounds)
    print("Empty geometries:", gdf.geometry.is_empty.sum())
    print("Null geometries:", gdf.geometry.isna().sum())
    if len(gdf) > 0:
        print("Area m2:", gdf.to_crs(TARGET_CRS).area.sum())


def clean_output_folder(folder):
    if os.path.exists(folder):
        shutil.rmtree(folder)
    os.makedirs(folder, exist_ok=True)


def check_saved_shapefile(output_base):
    expected_files = [".shp", ".shx", ".dbf", ".prj", ".cpg"]

    print("\n--- Saved shapefile components ---")
    for ext in expected_files:
        f = output_base + ext
        if os.path.exists(f):
            print(f"✅ {ext} exists | size: {os.path.getsize(f):,} bytes")
        else:
            print(f"❌ {ext} missing")

    saved = gpd.read_file(output_base + ".shp")

    check_gdf(saved, "Saved shapefile re-read from disk")

    return saved


# --------------------------------------------------
# 3. Read inputs
# --------------------------------------------------

large = gpd.read_file(LARGE_PATH)
subs = gpd.read_file(SUBS_PATH)

check_gdf(large, "Input large catchment")
check_gdf(subs, "Input contributing catchments")


# --------------------------------------------------
# 4. CRS handling
# --------------------------------------------------

if large.crs is None:
    large = large.set_crs(TARGET_CRS)

if subs.crs is None:
    subs = subs.set_crs(TARGET_CRS)

subs = subs.to_crs(large.crs)

large = large.to_crs(TARGET_CRS)
subs = subs.to_crs(TARGET_CRS)

check_gdf(large, "Large catchment after CRS conversion")
check_gdf(subs, "Contributing catchments after CRS conversion")

print("\nDo inputs overlap?")
print(large.unary_union.intersects(subs.unary_union))


# --------------------------------------------------
# 5. Fix geometries
# --------------------------------------------------

large["geometry"] = large.geometry.make_valid()
subs["geometry"] = subs.geometry.make_valid()

large = large[large.geometry.notna() & ~large.geometry.is_empty].copy()
subs = subs[subs.geometry.notna() & ~subs.geometry.is_empty].copy()


# --------------------------------------------------
# 6. Difference operation
# --------------------------------------------------

large_union = unary_union(large.geometry)
subs_union = unary_union(subs.geometry)

subs_union_buffered = subs_union.buffer(BUFFER_DISTANCE_M)

remaining_geom = large_union.difference(subs_union_buffered)

print("\n--- Difference geometry ---")
print("Type:", remaining_geom.geom_type)
print("Is empty:", remaining_geom.is_empty)
print("Area m2:", remaining_geom.area)
print("Bounds:", remaining_geom.bounds)


# --------------------------------------------------
# 7. Explode into separate polygons and keep largest
# --------------------------------------------------

remaining = gpd.GeoDataFrame(geometry=[remaining_geom], crs=TARGET_CRS)
remaining = remaining.explode(index_parts=False).reset_index(drop=True)

remaining = remaining[
    remaining.geometry.notna() &
    ~remaining.geometry.is_empty
].copy()

remaining["area_m2"] = remaining.area
remaining["area_km2"] = remaining["area_m2"] / 1_000_000

print("\n--- Remaining polygon parts ---")
print("Number of polygons:", len(remaining))
print(
    remaining[["area_m2", "area_km2"]]
    .sort_values("area_m2", ascending=False)
    .head(20)
)

remaining_main = (
    remaining
    .sort_values("area_m2", ascending=False)
    .head(1)
    .drop(columns=["area_m2", "area_km2"])
    .copy()
)


# --------------------------------------------------
# 8. Make ArcMap-safe output and save shapefile
# --------------------------------------------------

import os
import shutil

OUTPUT_FOLDER = "/dbfs/FileStore/AndyBeverton/AOI"
OUTPUT_NAME = "33035_minus_contributingcatchments"
OUTPUT_PATH = f"{OUTPUT_FOLDER}/{OUTPUT_NAME}.shp"

# Clean output folder
if os.path.exists(OUTPUT_FOLDER):
    shutil.rmtree(OUTPUT_FOLDER)

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Take largest polygon
largest_geom = (
    remaining
    .sort_values("area_m2", ascending=False)
    .geometry
    .iloc[0]
)

# Force simple GeoDataFrame: one integer field + geometry only
arcmap_safe = gpd.GeoDataFrame(
    {"id": [1]},
    geometry=[largest_geom],
    crs="EPSG:27700"
)

# Ensure polygon-only and valid
arcmap_safe["geometry"] = arcmap_safe.geometry.make_valid()
arcmap_safe = arcmap_safe.explode(index_parts=False).reset_index(drop=True)
arcmap_safe = arcmap_safe[arcmap_safe.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
arcmap_safe["id"] = range(1, len(arcmap_safe) + 1)

arcmap_safe.to_file(
    OUTPUT_PATH,
    driver="ESRI Shapefile"
)

print("Saved ArcMap-safe shapefile to:")
print(OUTPUT_PATH)

# Re-read check
saved = gpd.read_file(OUTPUT_PATH)

print("\nSaved check:")
print("Rows:", len(saved))
print("CRS:", saved.crs)
print("Geometry types:")
print(saved.geom_type.value_counts())
print("Bounds:", saved.total_bounds)
print("Area m2:", saved.to_crs("EPSG:27700").area.sum())

display(dbutils.fs.ls("dbfs:/FileStore/AndyBeverton/AOI/"))

# --------------------------------------------------
# 9. Check saved files by reading them back
# --------------------------------------------------

OUTPUT_BASE = f"{OUTPUT_FOLDER}/{OUTPUT_NAME}"
saved = check_saved_shapefile(OUTPUT_BASE)

print("\n--- Compare before and after save ---")
print("Before save rows:", len(remaining_main))
print("After save rows:", len(saved))
print("Before save CRS:", remaining_main.crs)
print("After save CRS:", saved.crs)
print("Before save bounds:", remaining_main.total_bounds)
print("After save bounds:", saved.total_bounds)
print("Before save area m2:", remaining_main.area.sum())
print("After save area m2:", saved.to_crs(TARGET_CRS).area.sum())


# --------------------------------------------------
# 10. Databricks download links
# --------------------------------------------------

download_base = f"/files/AndyBeverton/33035_minus_contributingcatchments_out/{OUTPUT_NAME}"

displayHTML(f"""
<h3>Download final shapefile components</h3>
<p>Download all of these into the same folder on your laptop:</p>
<ul>
  <li><a href="{download_base}.shp" target="_blank">{OUTPUT_NAME}.shp</a></li>
  <li><a href="{download_base}.shx" target="_blank">{OUTPUT_NAME}.shx</a></li>
  <li><a href="{download_base}.dbf" target="_blank">{OUTPUT_NAME}.dbf</a></li>
  <li><a href="{download_base}.prj" target="_blank">{OUTPUT_NAME}.prj</a></li>
  <li><a href="{download_base}.cpg" target="_blank">{OUTPUT_NAME}.cpg</a></li>
</ul>
""")


# --------------------------------------------------
# 11. Plot inputs and saved output
# --------------------------------------------------

fig = plt.figure(figsize=(18, 14))

ax1 = plt.subplot2grid((2, 3), (0, 0))
ax2 = plt.subplot2grid((2, 3), (0, 1))
ax3 = plt.subplot2grid((2, 3), (0, 2))
ax4 = plt.subplot2grid((2, 3), (1, 0), colspan=3)

large.plot(ax=ax1, facecolor="none", edgecolor="black", linewidth=1.5)
ax1.set_title("Input 1: 33035 large catchment")

subs.plot(ax=ax2, facecolor="none", edgecolor="red", linewidth=1.2, linestyle="--")
ax2.set_title("Input 2: contributing catchments")

saved.plot(ax=ax3, facecolor="none", edgecolor="green", linewidth=1.5)
ax3.set_title("Saved output: largest remaining polygon")

large.plot(ax=ax4, facecolor="lightgrey", edgecolor="black", linewidth=1.5, alpha=0.25)
subs.plot(ax=ax4, facecolor="red", edgecolor="red", linewidth=1.0, alpha=0.35, linestyle="--")
saved.plot(ax=ax4, facecolor="none", edgecolor="green", linewidth=2.0)

ax4.set_title("Combined check: original, removed catchments, and saved output")

legend_items = [
    Line2D([0], [0], color="black", linewidth=1.5, label="33035 original catchment"),
    Line2D([0], [0], color="red", linewidth=1.5, linestyle="--", label="Removed contributing catchments"),
    Line2D([0], [0], color="green", linewidth=2.0, label="Saved final output"),
]

ax4.legend(handles=legend_items, loc="best")

for ax in [ax1, ax2, ax3, ax4]:
    ax.set_aspect("equal")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.ticklabel_format(useOffset=False)
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

In [0]:
# ==================================================
# Remove contributing catchments from 33035 catchment
# Write shapefile safely for ArcMap
# ==================================================

import os
import glob
import shutil
import geopandas as gpd
import matplotlib.pyplot as plt

from shapely.ops import unary_union
from matplotlib.lines import Line2D


# --------------------------------------------------
# 1. User settings
# --------------------------------------------------

BASE_DIR = "/dbfs/FileStore/AndyBeverton"

LARGE_PATH = f"{BASE_DIR}/33035/33035.shp"
SUBS_PATH = f"{BASE_DIR}/Contributingcatchments/Contributing_Catchments.shp"

OUTPUT_FOLDER = f"{BASE_DIR}/AOI"
OUTPUT_NAME = "33035_minus_contributingcatchments"
OUTPUT_PATH = f"{OUTPUT_FOLDER}/{OUTPUT_NAME}.shp"

ALL_PARTS_FOLDER = f"{BASE_DIR}/ALL PARTS"
ALL_PARTS_PATH = f"{ALL_PARTS_FOLDER}/{OUTPUT_NAME}_ALL_PARTS.shp"

TARGET_CRS = "EPSG:27700"
BUFFER_DISTANCE_M = 10  # try 0, 5, 10, 25, 50


# --------------------------------------------------
# 2. Helper functions
# --------------------------------------------------

def check_gdf(gdf, name):
    print(f"\n--- {name} ---")
    print("Rows:", len(gdf))
    print("CRS:", gdf.crs)
    print("Geometry types:")
    print(gdf.geometry.geom_type.value_counts())
    print("Bounds:", gdf.total_bounds)
    print("Empty geometries:", gdf.geometry.is_empty.sum())
    print("Null geometries:", gdf.geometry.isna().sum())
    if len(gdf) > 0:
        print("Area m2:", gdf.to_crs(TARGET_CRS).area.sum())


def clean_folder(folder):
    if os.path.exists(folder):
        shutil.rmtree(folder)
    os.makedirs(folder, exist_ok=True)


def check_shapefile_files(folder, name):
    print(f"\n--- Shapefile files in {folder} ---")
    for ext in [".shp", ".shx", ".dbf", ".prj", ".cpg"]:
        path = f"{folder}/{name}{ext}"
        if os.path.exists(path):
            print(f"✅ {ext} exists | size: {os.path.getsize(path):,} bytes")
        else:
            print(f"❌ {ext} missing")


# --------------------------------------------------
# 3. Read inputs
# --------------------------------------------------

large = gpd.read_file(LARGE_PATH)
subs = gpd.read_file(SUBS_PATH)

check_gdf(large, "Input large catchment")
check_gdf(subs, "Input contributing catchments")


# --------------------------------------------------
# 4. CRS handling
# --------------------------------------------------

if large.crs is None:
    large = large.set_crs(TARGET_CRS)

if subs.crs is None:
    subs = subs.set_crs(TARGET_CRS)

subs = subs.to_crs(large.crs)

large = large.to_crs(TARGET_CRS)
subs = subs.to_crs(TARGET_CRS)

check_gdf(large, "Large catchment after CRS conversion")
check_gdf(subs, "Contributing catchments after CRS conversion")

print("\nDo inputs overlap?")
print(large.union_all().intersects(subs.union_all()))


# --------------------------------------------------
# 5. Fix geometries
# --------------------------------------------------

large["geometry"] = large.geometry.make_valid()
subs["geometry"] = subs.geometry.make_valid()

large = large[large.geometry.notna() & ~large.geometry.is_empty].copy()
subs = subs[subs.geometry.notna() & ~subs.geometry.is_empty].copy()


# --------------------------------------------------
# 6. Difference operation
# --------------------------------------------------

large_union = large.union_all()
subs_union = subs.union_all()

subs_union_buffered = subs_union.buffer(BUFFER_DISTANCE_M)

remaining_geom = large_union.difference(subs_union_buffered)

print("\n--- Difference geometry ---")
print("Type:", remaining_geom.geom_type)
print("Is empty:", remaining_geom.is_empty)
print("Area m2:", remaining_geom.area)
print("Bounds:", remaining_geom.bounds)


# --------------------------------------------------
# 7. Explode into separate polygons and keep largest
# --------------------------------------------------

remaining = gpd.GeoDataFrame(geometry=[remaining_geom], crs=TARGET_CRS)

remaining = remaining.explode(index_parts=False).reset_index(drop=True)

remaining = remaining[
    remaining.geometry.notna() &
    ~remaining.geometry.is_empty
].copy()

remaining["area_m2"] = remaining.area
remaining["area_km2"] = remaining["area_m2"] / 1_000_000

print("\n--- Remaining polygon parts ---")
print("Number of polygons:", len(remaining))
print(
    remaining[["area_m2", "area_km2"]]
    .sort_values("area_m2", ascending=False)
    .head(20)
)

largest_geom = (
    remaining
    .sort_values("area_m2", ascending=False)
    .geometry
    .iloc[0]
)

output_gdf = gpd.GeoDataFrame(
    {"id": [1]},
    geometry=[largest_geom],
    crs=TARGET_CRS
)

output_gdf["geometry"] = output_gdf.geometry.make_valid()
output_gdf = output_gdf.explode(index_parts=False).reset_index(drop=True)
output_gdf = output_gdf[
    output_gdf.geometry.notna() &
    ~output_gdf.geometry.is_empty &
    output_gdf.geom_type.isin(["Polygon", "MultiPolygon"])
].copy()

output_gdf["id"] = range(1, len(output_gdf) + 1)

check_gdf(output_gdf, "Final output before save")


# --------------------------------------------------
# 8. Save shapefile safely
# --------------------------------------------------

clean_folder(LOCAL_OUTPUT_FOLDER)
clean_folder(DBFS_OUTPUT_FOLDER)

# Write to local cluster disk first
output_gdf.to_file(
    LOCAL_OUTPUT_PATH,
    driver="ESRI Shapefile"
)

print("\nWritten locally to:")
print(LOCAL_OUTPUT_PATH)

check_shapefile_files(LOCAL_OUTPUT_FOLDER, OUTPUT_NAME)

# Re-read local output before copying to DBFS
local_saved = gpd.read_file(LOCAL_OUTPUT_PATH)
check_gdf(local_saved, "Local saved shapefile re-read")

# Copy all completed sidecar files to DBFS/FileStore
for file_path in glob.glob(f"{LOCAL_OUTPUT_FOLDER}/{OUTPUT_NAME}.*"):
    shutil.copy(file_path, DBFS_OUTPUT_FOLDER)
    print("Copied to DBFS:", os.path.basename(file_path))

print("\nCopied final shapefile to:")
print(DBFS_OUTPUT_FOLDER)

check_shapefile_files(DBFS_OUTPUT_FOLDER, OUTPUT_NAME)

# Re-read DBFS output
saved = gpd.read_file(DBFS_OUTPUT_PATH)
check_gdf(saved, "DBFS saved shapefile re-read")


# --------------------------------------------------
# 9. Download links
# --------------------------------------------------

download_base = f"/files/AndyBeverton/{OUTPUT_NAME}_out/{OUTPUT_NAME}"

displayHTML(f"""
<h3>Download final shapefile components</h3>
<p>Download all of these into the same folder on your laptop:</p>
<ul>
  <li><a href="{download_base}.shp" target="_blank">{OUTPUT_NAME}.shp</a></li>
  <li><a href="{download_base}.shx" target="_blank">{OUTPUT_NAME}.shx</a></li>
  <li><a href="{download_base}.dbf" target="_blank">{OUTPUT_NAME}.dbf</a></li>
  <li><a href="{download_base}.prj" target="_blank">{OUTPUT_NAME}.prj</a></li>
  <li><a href="{download_base}.cpg" target="_blank">{OUTPUT_NAME}.cpg</a></li>
</ul>
""")


# --------------------------------------------------
# 10. Plot inputs and saved output
# --------------------------------------------------

fig = plt.figure(figsize=(18, 14))

ax1 = plt.subplot2grid((2, 3), (0, 0))
ax2 = plt.subplot2grid((2, 3), (0, 1))
ax3 = plt.subplot2grid((2, 3), (0, 2))
ax4 = plt.subplot2grid((2, 3), (1, 0), colspan=3)

large.plot(ax=ax1, facecolor="none", edgecolor="black", linewidth=1.5)
ax1.set_title("Input 1: 33035 large catchment")

subs.plot(ax=ax2, facecolor="none", edgecolor="red", linewidth=1.2, linestyle="--")
ax2.set_title("Input 2: contributing catchments")

saved.plot(ax=ax3, facecolor="none", edgecolor="green", linewidth=1.5)
ax3.set_title("Saved output re-read from DBFS")

large.plot(ax=ax4, facecolor="lightgrey", edgecolor="black", linewidth=1.5, alpha=0.25)
subs.plot(ax=ax4, facecolor="red", edgecolor="red", linewidth=1.0, alpha=0.35, linestyle="--")
saved.plot(ax=ax4, facecolor="none", edgecolor="green", linewidth=2.0)

ax4.set_title("Combined check: original, removed catchments, and saved output")

legend_items = [
    Line2D([0], [0], color="black", linewidth=1.5, label="33035 original catchment"),
    Line2D([0], [0], color="red", linewidth=1.5, linestyle="--", label="Removed contributing catchments"),
    Line2D([0], [0], color="green", linewidth=2.0, label="Saved final output"),
]

ax4.legend(handles=legend_items, loc="best")

for ax in [ax1, ax2, ax3, ax4]:
    ax.set_aspect("equal")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.ticklabel_format(useOffset=False)
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

In [0]:
%sh
set -euo pipefail

SRC_BASE="/dbfs/FileStore/AndyBeverton"
DEST_BASE="/dbfs/FileStore/AndyBeverton/ZIPs_Download"
SUB_DIRECTORY="33035_minus_contributingcatchments_out"
WORKSPACE_DOMAIN="adb-7393756451346106.6.azuredatabricks.net"

SRC_DIR="${SRC_BASE}/${SUB_DIRECTORY}"
LOCAL_STAGING="/tmp/${SUB_DIRECTORY}"
LOCAL_ZIP="/tmp/${SUB_DIRECTORY}.zip"
FINAL_DEST="${DEST_BASE}/${SUB_DIRECTORY}.zip"

if [ ! -d "$SRC_DIR" ]; then
  echo "Source directory does not exist: $SRC_DIR"
  exit 1
fi

mkdir -p "$DEST_BASE"

rm -rf "$LOCAL_STAGING" "$LOCAL_ZIP"
mkdir -p "$LOCAL_STAGING"

# Copy without preserving permissions
cp -r "${SRC_DIR}/." "$LOCAL_STAGING/"

echo "Copied files to local staging:"
ls -lh "$LOCAL_STAGING"

# Create zip locally
cd "$LOCAL_STAGING"
zip -r -q "$LOCAL_ZIP" .

echo "Created local zip:"
ls -lh "$LOCAL_ZIP"

# Copy zip back to FileStore
cp "$LOCAL_ZIP" "$FINAL_DEST"

echo "Zip copied to:"
echo "$FINAL_DEST"

PUBLIC_URL="https://${WORKSPACE_DOMAIN}/files/AndyBeverton/ZIPs_Download/${SUB_DIRECTORY}.zip"

echo ""
echo "Download URL:"
echo "$PUBLIC_URL"